> **Historical notebook.** This is the exploratory scratchpad from the project's first two stages (environment bring-up and the first prompt/scoring experiments), kept as a record of how early decisions were made. Nothing imports from it and nothing here is current: every constant it defines was later moved into `config.py`, precisely because values living in an unexecuted notebook cell are invisible to tests and to review. For the current pipeline see `config.py` -> `hooks.py` -> `run.py` -> `analyze.py`.

# CS 2881R HW0 — J-space Ablation × Chain-of-Thought

**Model** `Qwen/Qwen3-4B` (bf16, MPS) · **Data** GSM8K → MATH-500 → AIME

Runs top to bottom on a clean kernel. Every cell is tagged with the milestone
it belongs to. Frozen logic lives in `scoring.py` / `analysis.py` and is
covered by `test_scoring.py` / `test_analysis.py`; this notebook is the
exploration record and the run driver, not the implementation.

| # | Milestone | Status |
|---|---|---|
| 1 | Load Qwen3-4B, generate text | done |
| 2 | One GSM8K problem, three ways | done |
| 3 | Answer extraction + scoring | done |
| **4** | **Eval loop, n=20, headroom verdict** | **HERE** |
| 5 | Hooks 101 — read one layer's activation | local |
| 6 | Trivial intervention — confirm output degrades | local |
| 7 | Real ablation — project out top-k workspace directions | GPU |
| 8 | Compose — eval loop × ablation + random-direction control | GPU |

**The quantity of interest** is one number:

```
interaction = (direct_intact − direct_ablated) − (cot_intact − cot_ablated)
```

Reporting the two drops separately invites "the ablation just broke
everything". The interaction is what separates internal/external
substitution from broad degradation.

---
## Setup

In [1]:
# [setup] Imports and module reload.
#
# Re-run this after ANY edit to scoring.py / analysis.py. Reloading beats
# restarting the kernel, which would cost an 8 GB model reload. Order
# matters: analysis imports names from scoring, so scoring reloads first.
#
# The `from ... import` lines are NOT optional after a reload -- names bound
# in this namespace still point at the old function objects otherwise.
import gc, importlib, json, os, platform, random, re, sys, time
import torch

import config, scoring, analysis
importlib.reload(config)
importlib.reload(scoring)
importlib.reload(analysis)

from scoring import (GOLD_FIELD, cond_name, dataset_fingerprint,
                     distinct_ngram_ratio, model_revision, provenance,
                     render_prompt, score, score_file, strip_think,
                     think_trace, unpack_cond, validate_gold)
from analysis import baseline_table, load_scores, observed_rho
# config.py IS the pre-registration. Caps, bands, k values and the
# problem sample live there so a change is a reviewable diff, and so a
# test can reach them. Nothing below may hand-type any of them.
from config import cap_for, problem_ids

from importlib.metadata import version
for p in ("torch", "transformers", "datasets", "math-verify",
          "latex2sympy2_extended", "sympy", "numpy"):
    print(f"{p:24} {version(p)}")
print(f"\npython {sys.version.split()[0]} | {platform.machine()} "
      f"| {platform.system()}")

# If memory pressure climbs across a long session, `%reset -f out` clears the
# Out[] cache, which can otherwise pin large tensors.

torch                    2.13.0
transformers             5.14.1
datasets                 5.0.1
math-verify              0.9.0
latex2sympy2_extended    1.11.0
sympy                    1.14.0
numpy                    2.5.1

python 3.14.6 | arm64 | Darwin


In [2]:
# [M1] Load Qwen3-4B. ~8 GB in bf16. Idempotent -- safe to re-run.
#
# The checkpoint is the HYBRID Qwen3-4B, not the later -Thinking-2507 /
# -Instruct-2507 splits. Only the hybrid exposes the `enable_thinking` toggle
# in apply_chat_template, which the whole three-condition design depends on.
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-4B"
device = "mps"          # cuda = NVIDIA, mps = Apple silicon, cpu = always works

for _n in ("model", "tok"):
    if _n in globals():
        del globals()[_n]
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)
model.eval()

c = model.config
# "Qwen/Qwen3-4B" names a BRANCH, not a snapshot. Resolve the commit
# here, where the weights are loaded -- not in the manifest cell hours
# later. The GPU run spans several cells and may straddle an instance
# restart, and a checkpoint that resolves differently between the first
# cell and the last is a confound with no signature in the data.
MODEL_REVISION = model_revision(model)
assert MODEL_REVISION, "could not resolve the checkpoint commit"
print(f"revision {MODEL_REVISION}")
print(f"torch {torch.__version__} | device={device} | {platform.machine()}")
print(f"LAYERS={c.num_hidden_layers}  d_model={c.hidden_size}  "
      f"vocab={c.vocab_size}")

# Layer-band translation for Half B, on 36 layers (indices 0-35):
#   workspace band  ~ 14-33   (paper's 0.38-0.92 depth on a 0-100 scale)
#   sensory         ~ 0-12      motor ~ 33-35
# Two problems to state in the report: (a) the band's top collides with the
# motor region; (b) Claude models are far deeper, so the paper's 55-point
# band compresses into ~20 Qwen layers -- and since ablation STRENGTH is
# defined as band width, our granularity is much coarser than theirs.

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

torch 2.13.0 | device=mps | arm64
LAYERS=36  d_model=2560  vocab=151936


In [3]:
# [M4] Freeze decoding.
#
# Qwen3 ships generation_config with do_sample=True (temp 0.6, top_p 0.95).
# An explicit do_sample=False per call overrides it -- but any generate() in
# Half B that forgets the kwarg would silently SAMPLE, producing a broken
# comparison with no error and no traceback. Close it at the source.
#
# Verified greedy: two identical calls returned identical text.
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_k = None
model.generation_config.top_p = None
assert model.generation_config.do_sample is False
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643
}



---
## Milestone 1 — load and generate

Establishes the throughput ceiling that every later time estimate is built on.

In [4]:
# [M1] First generation + steady-state throughput.
#
# Measured on M4 / 16 GB: ~7.7 tok/s. Memory-bandwidth bound, physical ceiling
# ~15 tok/s -- no software change will beat this by much, which is why the
# only permitted speed lever later is BATCHING, and why vLLM is ruled out
# (it compiles the graph, leaving no clean hook point on the residual stream).
#
# min_new_tokens forces generation past EOS so we measure 200 tokens of real
# work rather than 12.
probe = tok(render_prompt(tok, "In one sentence, what is 17 * 3?", thinking=False),
            return_tensors="pt").to(device)

with torch.no_grad():                                    # warm-up, discarded
    _ = model.generate(**probe, max_new_tokens=8, do_sample=False)

t0 = time.time()
with torch.no_grad():
    out = model.generate(**probe, max_new_tokens=200, min_new_tokens=200,
                         do_sample=False)
dt = time.time() - t0
n = out.shape[1] - probe.input_ids.shape[1]
print(tok.decode(out[0, probe.input_ids.shape[1]:], skip_special_tokens=True)[:200])
print(f"\n>>> {n} tokens in {dt:.1f}s = {n/dt:.1f} tok/s")

# KV cache ~144 KB/token: fine to ~8k tokens on 16 GB, ~32k will not fit.
# That is the AIME risk, and it arrives at the same time as the time cost.

17 multiplied by 3 is 51.</think>

17 * 3 = 51.</think>
</think>

17 * 3 = 51.</think>
</think>

17 * 3 = 51. ✅</think>
</think>

17 * 3 = 51. ✅</think>
</think>

17 * 3 = 51. ✅</think>
</think>

17 *

>>> 200 tokens in 20.7s = 9.7 tok/s


---
## Milestone 2 — one problem, three ways

The finding that forced **three** conditions instead of two:
`enable_thinking=False` buys *no `<think>` block*, not *no chain of thought*.
The model still writes `48/2 = 24, 48+24 = 72` as ordinary prose.

A "direct" condition that externalises onto the page is not direct, and would
have collapsed the interaction term via a prompt bug rather than a null result.

> **Historical.** The direct condition below is the original
> instruction-only version. Milestone 4 later showed it leaks on hard
> problems. The live definition is `CONDS`, further down.

In [5]:
# [M2] Streaming comparison helper. Uses render_prompt so it cannot drift
# from the prompts the real runs use.
from transformers import TextStreamer

PROBLEM = ("Natalia sold clips to 48 of her friends in April, and then she sold "
           "half as many clips in May. How many clips did Natalia sell altogether "
           "in April and May?")
DIRECT_SUFFIX = ("\n\nRespond with only the final numeric answer and nothing else. "
                 "Do not show any reasoning.")


def run(label, question, thinking, suffix="", prefill="", max_new=512):
    text = render_prompt(tok, question, thinking, suffix, prefill)
    ins = tok(text, return_tensors="pt").to(device)
    print("=" * 70)
    print(f"{label} | thinking={thinking} | prefill={prefill!r} | cap={max_new}")
    print("--- prompt tail ---"); print(repr(text[-120:]))
    print("--- streaming ---")
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ins, max_new_tokens=max_new, do_sample=False,
                             streamer=TextStreamer(tok, skip_prompt=True,
                                                   skip_special_tokens=False))
    dt = time.time() - t0
    n = out.shape[1] - ins.input_ids.shape[1]
    print(f"\n>>> {n} tok | {dt:.0f}s | {n/dt:.1f} tok/s | hit_cap={n >= max_new}")

In [6]:
# [M2] Cheap first, so a surprise surfaces in seconds rather than minutes.
run("C: thinking off + direct instruction", PROBLEM, thinking=False,
    suffix=DIRECT_SUFFIX, max_new=128)
run("B: thinking off", PROBLEM, thinking=False, max_new=512)
run("A: full CoT", PROBLEM, thinking=True, max_new=1024)

# Measured, one GSM8K problem: A 738 tok / 104 s, B 94 tok / 14 s, C 4 tok / 2 s.
# NOTE: `PROBLEM` (Natalia) is a 2-step question. C's clean 4-token answer here
# is REAL but not representative -- see the difficulty probe in Milestone 4.

C: thinking off + direct instruction | thinking=False | prefill='' | cap=128
--- prompt tail ---
'he final numeric answer and nothing else. Do not show any reasoning.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- streaming ---
144<|im_end|>

>>> 4 tok | 2s | 2.6 tok/s | hit_cap=False
B: thinking off | thinking=False | prefill='' | cap=512
--- prompt tail ---
'in May. How many clips did Natalia sell altogether in April and May?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
--- streaming ---
Natalia sold clips to **48 friends** in **April**.

In **May**, she sold **half as many** clips as in April:

$$
\frac{48}{2} = 24
$$

Now, add the number of clips sold in both months:

$$
48 + 24 = 72
$$

**Answer:** Natalia sold **72 clips** altogether in April and May.<|im_end|>

>>> 94 tok | 10s | 9.5 tok/s | hit_cap=False
A: full CoT | thinking=True | prefill='' | cap=1024
--- prompt tail ---
'half as many clips in May. How many clips did Natalia sell altogether in 

---
## Milestone 3 — answer extraction and scoring

The hand-rolled `norm` / `NUM` / `extract_answer` / `strip_think` regex scorer
that used to live here has been **deleted**, not commented out. It is
superseded by `scoring.py`, and leaving it live in this namespace meant two
real hazards:

1. it *shadowed* `scoring.strip_think` depending on cell execution order, and
   the notebook copy still had the `<|im_end|>` bug (delete instead of
   truncate);
2. its last-number fallback is exactly the length-scaling leniency that
   `extraction_mode="first_match"` was chosen to remove — leniency that grows
   with output length biases the CoT cells upward.

Scoring policy, pre-registered: **four outcomes** (correct / incorrect /
incomplete / unparsed, plus `error`), headline accuracy counts everything that
is not `correct` as wrong, and the outcome composition is always reported
alongside. An accuracy collapse made of `incomplete` is a termination failure,
not a reasoning failure.

In [7]:
# [M3] Dataset + gold pre-flight.
import datasets

ds = datasets.load_dataset("openai/gsm8k", "main")
test = ds["test"]
print(ds)
print("datasets", datasets.__version__)          # pin this in the README
print("fields:", list(test[0].keys()))
print("gold[0]:", repr(GOLD_FIELD["gsm8k"](test[0])))

# Free pre-flight, no GPU: can we parse the dataset's OWN answers? Run this
# before spending an hour generating.
bad = validate_gold("gsm8k", [test[i] for i in range(200)])
print("unparseable golds in first 200:", bad)
assert not bad, bad

# Split-level fingerprint. The per-run fingerprint below adds a content
# hash over the sampled ids, which is what would catch a release that
# keeps the row count and edits a problem statement.
DS_ARGS = dict(path="openai/gsm8k", name="main", split="test")
print(dataset_fingerprint(test, **DS_ARGS))

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
datasets 5.0.1
fields: ['question', 'answer']
gold[0]: '18'
unparseable golds in first 200: []


In [10]:
# [M3] The frozen scorers are tested, not trusted.
#
# 35 scoring cases + 9 unit tests, plus recovery / coverage / false-positive
# validation of the analysis layer. Verified byte-identical on macOS arm64 and
# Linux x86, and every fix has a regression that goes red when reverted.
!python test_scoring.py --quiet

ok   strip_think
ok   think_trace
ok   unwrap_markdown
ok   normalisation is fallback-only
ok   degeneracy  clean=1.00 loop=0.03 cot(body=1.00 trace=0.03)
ok   condition naming grid
ok   unpack_cond
ok   render_prompt + prefill changes the fingerprint
ok   provenance  math-verify=0.9.0 git=aa94673-dirty

ALL PASS (35 scoring cases + 9 unit tests)


In [11]:
# [M3] Analysis layer: parameter recovery, CI coverage, false-positive rate.
#
# --no-demo runs the assertions only. Drop the flag to also print four worked
# examples: a clean n=100 run, the underpowered n=20 case, a contaminated run
# where the cap and degeneracy warnings both fire, and the main analysis
# beside its random-direction control.
#
# Do NOT pipe this to `head`. head exits after N lines and closes the pipe,
# killing the still-running process on its next print. (`tail` reads to EOF,
# so piping to tail is safe -- that asymmetry is why one of these cells used
# to crash and the other did not.) Both scripts now survive it anyway.
!python test_analysis.py --no-demo

ok   mcnemar exact  b=3 c=2 p=1.0000
ok   recovery       truth=+20% est=+20.7% err=0.007 (n=4000)
ok   coverage       94% of 95% CIs contain truth (n=100, mean width 27%)
ok   false positive  6% when true interaction is 0
ok   complete cases  paired on 8/10, dropped [3, 7]
ok   empty cell     raises instead of returning a NaN null
ok   no overlap     raises instead of pairing on nothing
ok   paired table    all-records acc=75% vs paired 50%; report now prints the paired one
ok   p floor        p=1.00e-04 flagged, not printed as 0.000
ok   cap warnings   2 raised on a 30% direct-only cap
ok   duplicates    raise instead of last-write-wins
       rho: target=0.00 measured=0.014
       rho: target=0.30 measured=0.298
       rho: target=0.50 measured=0.496
       rho: target=0.75 measured=0.757
ok   observed_rho    recovers the correlation power.py assumes

ALL PASS



---
## Milestone 4 — baseline eval, n = 20

**Goal:** establish `cot_intact` and `direct_intact` accuracy, and decide
whether the experiment has headroom.

### The three conditions

Named from a grid (`scoring.LEVELS` × `scoring.STATES`) so a typo cannot reach
the analysis, and so these baseline records slot into the final 2×2 with no
renaming. A condition spec is `(enable_thinking, suffix, prefill)`.

| condition | thinking | prefill | role |
|---|---|---|---|
| `cot_intact` | True | — | full externalisation |
| `nothink_intact` | False | — | middle rung; leaks prose reasoning |
| `direct_intact` | False | `\boxed{` | **the direct condition** |

### Why the direct condition prefills instead of instructing

Asking a 4B model not to reason is instruction-following, and compliance
**degrades as problems get harder** — the same axis as the research question.
Leakage would then be inseparable from a real difficulty effect. Prefilling is
mechanical, so compliance does not vary with difficulty, and it makes the
prefill forward pass unambiguously where the computation lives (which is
exactly where Half B's ablation has to fire).

In [16]:
# [M4] Live condition definitions. Everything downstream reads these.
DIRECT_SUFFIX = ("\n\nRespond with only the final numeric answer and nothing else. "
                 "Do not show any reasoning.")
BOX = "\\boxed{"

# name -> (enable_thinking, suffix, prefill)
CONDS = {
    cond_name("cot",     "intact"): (True,  "",            ""),
    cond_name("nothink", "intact"): (False, "",            ""),
    cond_name("direct",  "intact"): (False, DIRECT_SUFFIX, BOX),
}
for name, spec in CONDS.items():
    th, sfx, pre = unpack_cond(spec)
    print(f"{name:16} think={th!s:5} cap={cap_for(name):<5} prefill={pre!r}")
    print("    ", repr(render_prompt(tok, "PROBE", th, sfx, pre)[-110:]))

# CHECK the printed tails:
#   direct_intact  must end   ...<think>\n\n</think>\n\n\boxed{
#   cot_intact     must NOT contain a pre-filled </think>

cot_intact       think=True  cap=3072  prefill=''
     '<|im_start|>user\nPROBE<|im_end|>\n<|im_start|>assistant\n'
nothink_intact   think=False cap=512   prefill=''
     '<|im_start|>user\nPROBE<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
direct_intact    think=False cap=32    prefill='\\boxed{'
     'answer and nothing else. Do not show any reasoning.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n\\boxed{'


### Evidence for the prefill decision

Run once. `boxed` is the candidate, `none_control` is the old instruction-only
prompt, probed at three difficulty levels (GSM8K annotates one `<<...>>` per
arithmetic step, which is the right axis because the failure mode is leakage
that *grows* with difficulty).

Measured result:

| difficulty | `boxed` | `none_control` |
|---|---|---|
| easy (0 steps) | 4 tok, correct | 7 tok, correct |
| mid (3 steps) | 4 tok, wrong | 3 tok, wrong |
| **hard (7 steps)** | **6 tok, wrong** | **255 tok, correct** |

The control grabbed the page on the hard problem *and that is why it got it
right*. That is the hypothesis appearing by accident — and the confound the
prefill removes.

`boxed` getting 1/3 right is the **desired** outcome, not a failure. Doing
multi-step arithmetic with no scratchpad is supposed to be hard; a direct
condition that aced everything would leave the ablation nothing to bite on
(the ">80% is also fatal" row of the decision rule).

In [17]:
# [M4] Difficulty probe. ~90 s, nearly all of it none_control.
def n_steps(r):
    return len(re.findall(r"<<", r["answer"]))


cand = sorted((n_steps(test[i]), i) for i in range(200))
PROBS = [("easy", cand[0][1], cand[0][0]),
         ("mid",  cand[len(cand) // 2][1], cand[len(cand) // 2][0]),
         ("hard", cand[-1][1], cand[-1][0])]
print("probe problems:", [(l, i, f"{s} steps") for l, i, s in PROBS], "\n")

PROBE_CAP = 256


def probe_direct(i, prefill, suffix=DIRECT_SUFFIX, cap=PROBE_CAP):
    ins = tok(render_prompt(tok, test[i]["question"], False, suffix, prefill),
              return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**ins, max_new_tokens=cap, do_sample=False)
    n = int(out.shape[1] - ins.input_ids.shape[1])
    raw = prefill + tok.decode(out[0, ins.input_ids.shape[1]:],
                               skip_special_tokens=False)
    return raw, n


print(f"{'variant':14}{'prob':>6}{'tok':>5}{'nums':>6}{'ops':>5}{'outcome':>11}  body")
for name, pre in (("boxed", BOX), ("none_control", "")):
    for lab, i, _ in PROBS:
        raw, n = probe_direct(i, pre)
        gold = GOLD_FIELD["gsm8k"](test[i])
        outcome, _ = score(raw, gold, hit_cap=(n >= PROBE_CAP), thinking=False)
        body = strip_think(raw)
        print(f"{name:14}{lab:>6}{n:>5}"
              f"{len(re.findall(r'[0-9]+(?:.[0-9]+)?', body)):>6}"
              f"{len(re.findall(r'[=+*/-]', body)):>5}{outcome:>11}  {body[:50]!r}")
    print()

probe problems: [('easy', 24, '0 steps'), ('mid', 68, '3 steps'), ('hard', 177, '7 steps')] 

variant         prob  tok  nums  ops    outcome  body
boxed           easy    4     1    0    correct  '\\boxed{26}'
boxed            mid    4     1    0  incorrect  '\\boxed{12}'
boxed           hard    6     1    0  incorrect  '\\boxed{1000}'

none_control    easy    7     1    0    correct  '$26.00'
none_control     mid    3     1    0  incorrect  '12'
none_control    hard  255    39   56    correct  "To solve this, we'll break down Zaid's monthly sal"



### Smoke test

One problem, all three conditions, through the real `score_file` path.
~3.5 minutes, essentially all CoT.

**Five checks, in priority order:**

1. `cot_intact` RAW contains `</think>` and TRACE is non-zero — if TRACE is 0
   words the degeneracy detector is dead, because looping lives inside the
   trace and the post-trace body is too short to measure.
2. `direct_intact` BODY is `\boxed{N}` with no arithmetic — any working shown
   and the direct condition is invalid.
3. `nothink_intact` BODY *does* contain arithmetic — this is the observation
   that justifies three conditions.
4. `hit_cap` False everywhere.
5. Nothing `unparsed`.

In [18]:
# [M4] Smoke test: generate.
os.makedirs("runs", exist_ok=True)
SMOKE = "runs/smoke.jsonl"
if os.path.exists(SMOKE):
    os.remove(SMOKE)        # a stale file holds the OLD leaky direct answer

i = 0
for cond, spec in CONDS.items():
    think, suffix, prefill = unpack_cond(spec)
    cap = cap_for(cond)
    ins = tok(render_prompt(tok, test[i]["question"], think, suffix, prefill),
              return_tensors="pt").to(device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ins, max_new_tokens=cap, do_sample=False)
    n = int(out.shape[1] - ins.input_ids.shape[1])
    # Store prefill + generated: the model emits "18}", and scoring
    # "\boxed{18}" is clean while scoring "18}" asks math-verify to guess.
    rec = dict(id=i, cond=cond, seed=0,
               raw=prefill + tok.decode(out[0, ins.input_ids.shape[1]:],
                                        skip_special_tokens=False),
               gold=GOLD_FIELD["gsm8k"](test[i]), n_tok=n,
               hit_cap=bool(n >= cap), secs=round(time.time() - t0, 1))
    with open(SMOKE, "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{cond:16}{n:5d} tok {rec['secs']:6.1f}s")

cot_intact       1580 tok  182.4s
nothink_intact    242 tok   26.9s
direct_intact       4 tok    0.7s


In [19]:
# [M4] Smoke test: score + hand-read. Scoring is ~2 ms, generation was minutes;
# they are deliberately decoupled so a scoring change is a diff, not a re-run.
score_file(SMOKE, "runs/smoke_scores.jsonl", CONDS,
           manifest_path="runs/smoke_manifest.json", tokenizer=tok,
           caps=config.CAPS, seed=config.SEED, model_name=MODEL,
           model_revision=MODEL_REVISION,
           dataset_fingerprints={"gsm8k": dataset_fingerprint(
               test, [0], **DS_ARGS)},
           gen_config=model.generation_config.to_dict())
baseline_table(load_scores("runs/smoke_scores.jsonl"), list(CONDS))

for r in [json.loads(l) for l in open(SMOKE)]:
    print("=" * 72)
    print(r["cond"], "| gold", r["gold"])
    print("RAW  :", repr(r["raw"][:200]))
    print("BODY :", repr(strip_think(r["raw"])[:150]))
    print("TRACE:", len(think_trace(r["raw"]).split()), "words")

print("\n--- manifest ---")
print(open("runs/smoke_manifest.json").read()[:700])

scored 3 records -> runs/smoke_scores.jsonl
  manifest -> runs/smoke_manifest.json
cell                n    acc  corr  inc trunc  unp  err   cap%  degen%  norm%    tok
cot_intact          1 100.0%     1    0     0    0    0     0%      0%     0%   1580
nothink_intact      1 100.0%     1    0     0    0    0     0%      0%     0%    242
direct_intact       1   0.0%     0    1     0    0    0     0%      0%     0%      4
cot_intact | gold 18
RAW  : "<think>\nOkay, let's see. Janet has a bunch of ducks that lay eggs every day. The problem says they lay 16 eggs per day. So first, I need to figure out how many eggs she uses for breakfast and how many"
BODY : "Janet's ducks lay 16 eggs per day. She uses 3 eggs for breakfast and 4 eggs to bake muffins for her friends. \n\nFirst, we calculate the total number of"
TRACE: 967 words
nothink_intact | gold 18
RAW  : "Let's break down the problem step by step.\n\n---\n\n### **Step 1: Total eggs laid per day**\nJanet’s ducks lay **16 eggs per day**.\

### The n = 20 baseline run

> **Gate: do not run until all five smoke checks pass.** ~71 minutes, and 63
> of those are CoT. A bug found after the expensive cell costs the whole hour.

Resumable — re-running skips `(id, cond)` pairs already on disk, so an
interrupted run continues rather than duplicating. Duplicates would be caught
by `to_matrix` anyway, but not until analysis time.

Conditions run **cheapest first**: direct (~1 s each) and nothink (~25 s each)
finish in about nine minutes, so a mistake surfaces before the CoT block
starts.

In [20]:
# [M4] Baseline runner. Resumable, cheap-first.
#
# Ids come from config.problem_ids, not random.sample. They are DIFFERENT ids:
# config.py documents why random.sample's nesting on GSM8K is a CPython
# `setsize` accident that breaks on MATH-500 (18 of 20 survive), and the
# pilot/run comparability of every number in the report rests on that nesting.
#
# So this is a different sample from the n=20 pilot in gsm8k_baseline.jsonl,
# and the output path carries N to say so. Appending a new id set to the pilot
# file would leave one file describing two samples, with nothing to warn you.
N = 20
SEED = config.SEED
OUT = f"runs/gsm8k_n{N}.jsonl"
os.makedirs("runs", exist_ok=True)

# Fails loudly if this cell is reading a stale hand-typed CAPS instead of the
# pre-registration. Decision 33 raised the direct cap 32 -> 128 and nothing
# else in the notebook would notice a half-finished edit.
assert cap_for("direct_intact") == 128, "not reading config.CAPS"

ids = problem_ids(N, len(test))
print("problem ids:", ids)

done = set()
if os.path.exists(OUT):
    for line in open(OUT):
        r = json.loads(line)
        done.add((r["id"], r["cond"]))
    stale = {i for i, _ in done} - set(ids)
    assert not stale, (f"{OUT} holds ids outside this sample: {sorted(stale)}. "
                       f"Resuming would mix two samples in one file.")
    print(f"resuming: {len(done)} records already on disk")

order = sorted(CONDS, key=cap_for)                    # cheap conditions first
t_start = time.time()
for cond in order:
    think, suffix, prefill = unpack_cond(CONDS[cond])
    cap = cap_for(cond)
    for i in ids:
        if (i, cond) in done:
            continue
        ins = tok(render_prompt(tok, test[i]["question"], think, suffix, prefill),
                  return_tensors="pt").to(device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**ins, max_new_tokens=cap, do_sample=False)
        n = int(out.shape[1] - ins.input_ids.shape[1])
        rec = dict(id=i, cond=cond, seed=SEED,
                   raw=prefill + tok.decode(out[0, ins.input_ids.shape[1]:],
                                            skip_special_tokens=False),
                   gold=GOLD_FIELD["gsm8k"](test[i]), n_tok=n,
                   hit_cap=bool(n >= cap), secs=round(time.time() - t0, 1))
        with open(OUT, "a") as f:
            f.write(json.dumps(rec) + "\n")
        print(f"{cond:16} id={i:<5} {n:5d} tok {rec['secs']:6.1f}s", flush=True)
print(f"\ntotal {(time.time() - t_start) / 60:.1f} min")


problem ids: [82, 194, 285, 286, 447, 513, 530, 577, 621, 733, 788, 829, 861, 976, 995, 1033, 1047, 1090, 1194, 1266]
direct_intact    id=82        5 tok    1.6s
direct_intact    id=194       3 tok    0.5s
direct_intact    id=285       4 tok    0.5s
direct_intact    id=286       3 tok    0.6s
direct_intact    id=447       3 tok    0.6s
direct_intact    id=513       5 tok    0.8s
direct_intact    id=530       4 tok    0.7s
direct_intact    id=577       4 tok    0.8s
direct_intact    id=621       6 tok    0.9s
direct_intact    id=733       3 tok    0.5s
direct_intact    id=788       5 tok    0.7s
direct_intact    id=829       7 tok    0.9s
direct_intact    id=861       4 tok    0.6s
direct_intact    id=976       5 tok    0.9s
direct_intact    id=995       4 tok    0.7s
direct_intact    id=1033      3 tok    0.6s
direct_intact    id=1047      3 tok    0.6s
direct_intact    id=1090      4 tok    0.7s
direct_intact    id=1194      6 tok    0.9s
direct_intact    id=1266      5 tok    0.7s
no

In [21]:
# [M4] Score the baseline and write the manifest. Instant.
rows = score_file(OUT, OUT.replace(".jsonl", "_scores.jsonl"), CONDS,
                  manifest_path=OUT.replace(".jsonl", "_manifest.json"),
                  tokenizer=tok,
                  caps=config.CAPS, seed=SEED, model_name=MODEL,
                  model_revision=MODEL_REVISION,
                  dataset_fingerprints={"gsm8k": dataset_fingerprint(
                      test, ids, **DS_ARGS)},
                  gen_config=model.generation_config.to_dict())
baseline_table(rows, list(CONDS))

# report() needs all four 2x2 cells and will raise on baseline-only data;
# baseline_table is the right call here.

scored 60 records -> runs/gsm8k_scores.jsonl
  manifest -> runs/gsm8k_manifest.json
cell                n    acc  corr  inc trunc  unp  err   cap%  degen%  norm%    tok
cot_intact         20  90.0%    18    1     1    0    0     5%      0%     0%   1385
nothink_intact     20  90.0%    18    2     0    0    0     0%      0%     0%    220
direct_intact      20  45.0%     9   11     0    0    0     0%      0%     0%      4


{'cot_intact': {'cond': 'cot_intact',
  'n': 20,
  'acc': 0.9,
  'correct': 18,
  'incorrect': 1,
  'incomplete': 1,
  'unparsed': 0,
  'error': 0,
  'hit_cap': 0.05,
  'degenerate': 0.0,
  'normalized': 0.0,
  'mean_tok': 1385.0},
 'nothink_intact': {'cond': 'nothink_intact',
  'n': 20,
  'acc': 0.9,
  'correct': 18,
  'incorrect': 2,
  'incomplete': 0,
  'unparsed': 0,
  'error': 0,
  'hit_cap': 0.0,
  'degenerate': 0.0,
  'normalized': 0.0,
  'mean_tok': 219.75},
 'direct_intact': {'cond': 'direct_intact',
  'n': 20,
  'acc': 0.45,
  'correct': 9,
  'incorrect': 11,
  'incomplete': 0,
  'unparsed': 0,
  'error': 0,
  'hit_cap': 0.0,
  'degenerate': 0.0,
  'normalized': 0.0,
  'mean_tok': 4.3}}

In [22]:
# [M4] Measure rho for the power analysis instead of guessing it.
#
# power.py's `rho` is the problem-difficulty correlation across cells, on the
# LATENT scale (analysis.observed_rho returns a tetrachoric estimate, not a
# Pearson phi -- dichotomising attenuates, so a latent 0.50 shows up as 0.33).
# It is load-bearing: at n=150 power runs 78% at rho=0 and 96% at rho=0.75.
import numpy as np

by = {c: {r["id"]: r["correct"] for r in rows if r["cond"] == c} for c in CONDS}
pair_ids = sorted(set.intersection(*(set(d) for d in by.values())))
mat = {c: np.array([by[c][i] for i in pair_ids], float) for c in CONDS}
for c in CONDS:
    print(f"  {c:16} acc={mat[c].mean():.0%}")

rho = observed_rho(mat)
if np.isnan(rho):
    # Every pair had a cell at 0% or 100%, where the latent threshold is
    # infinite and rho is unidentified. Plausible at n=20 if cot_intact
    # sweeps. Fall back, and SAY SO in the pre-registration.
    rho_arg = 0.50
    print(f"\npaired on {len(pair_ids)} problems | rho UNDEFINED "
          f"(a cell is at 0% or 100%) -- falling back to {rho_arg}")
else:
    rho_arg = rho
    print(f"\npaired on {len(pair_ids)} problems | measured rho = {rho:.3f}")

# Paste-ready. The two ablated numbers are GUESSES until Milestone 8 -- the
# defaults below assume a 30-point direct drop and a 10-point CoT drop.
d, ctx = mat["direct_intact"].mean(), mat["cot_intact"].mean()
print(f"\n  python power.py {d:.2f} {max(d - 0.30, 0.0):.2f} "
      f"{ctx:.2f} {max(ctx - 0.10, 0.0):.2f} --rho {rho_arg:.2f} --loop 0.15")
print("\n  --loop models degenerate looping in the ablated CoT cell. The clean")
print("  column is optimistic about a failure mode we already know happens:")
print("  15% looping turns a true +20pt interaction into +8pt, and power at")
print("  n=150 falls from 94% to 22%. Pre-register against the --loop column.")

  cot_intact       acc=90%
  nothink_intact   acc=90%
  direct_intact    acc=45%

paired on 20 problems | measured rho = -0.332

  python power.py 0.45 0.15 0.90 0.80 --rho -0.33 --loop 0.15

  --loop models degenerate looping in the ablated CoT cell. The clean
  column is optimistic about a failure mode we already know happens:
  15% looping turns a true +20pt interaction into +8pt, and power at
  n=150 falls from 94% to 22%. Pre-register against the --loop column.


### Decision rule — commit before looking

| `direct_intact` lands | verdict |
|---|---|
| **35 – 65 %** | healthy headroom → proceed to Milestone 5 |
| **< 15 %** | floor problem. Soften toward `nothink`, or restrict to problems it solves cleanly |
| **> 80 %** | also fatal. Either still leaking, or GSM8K is memorised well enough that no internal multi-step work happens — ablation would have nothing to bite on |

n = 20 locates a baseline to about ±20 points and is **useless** for
estimating an interaction. Success here does not imply the interaction is
measurable at this n.

**Also hand-read 2–3 `direct_intact` generations.** If any leaked reasoning
despite the prefill, redesign before building anything on top.

**Two secondary predictions**, recorded as free tests:

- `norm%` high for `direct_intact`, ~0 for `cot_intact` — a model told to emit
  only a number reaches for `**72**`. *(Now partly moot: the `\boxed{`
  prefill supplies the delimiter, so this may read 0 across the board.)*
- `degen%` ~0 on `cot_intact` — this quietly settles the greedy-vs-sampling
  question. Qwen advises against greedy in thinking mode because of repetition
  loops, and `distinct10_trace` measures precisely that. Zero means keep
  `do_sample=False` and keep determinism, with data behind the choice.

### After Milestone 4

Re-run `power.py` with the measured `direct_intact`, `cot_intact` and `rho`,
pick n, and **commit the falsification criterion before running anything
ablated**:

> *We reject substitution if the 95% paired-bootstrap CI on
> (direct_intact − direct_ablated) − (cot_intact − cot_ablated) includes zero
> at n = [N], which our power analysis shows resolves an interaction of ≥[X]
> points with 80% power.*

Note the mirror-image trap: an underpowered *null* is not an honest negative,
and an underpowered *positive* has an inflated magnitude. At n = 20 with a
true +20 % interaction, the simulation returns +35 % when it detects at all.
Report the interval as the headline; the point estimate is secondary.